# MarianMT: conversational machine translation

Run the cells in order (Jupyter or Google Colab; CPU works, CUDA is used when available).
The first run requires internet access to install packages and download two public models.
No Hugging Face key or Google Drive mount is required.

This notebook translates ten English inputs into French with `Helsinki-NLP/opus-mt-en-fr`,
compares beam widths **1, 2, 4, 8**, returns **three ranked hypotheses** with beam widths
**4 and 8**, and compares **English → French, Spanish, Italian, Portuguese** using
`Helsinki-NLP/opus-mt-en-ROMANCE`.

The original input cell contained **nine assignments**, each overwriting the preceding one.
We retain those nine entries, including repeats, and add a second “Long time no see!”
as input 10 to complete the repeated five-sentence set. There are ten inputs and five unique texts.

In [1]:
%pip install -q "torch>=2.6" "transformers>=4.45,<5" sentencepiece sacremoses pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 78.9 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 78.8 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import gc
import time
from pathlib import Path

import pandas as pd
import torch
import transformers
from IPython.display import display
from transformers import MarianMTModel, MarianTokenizer, set_seed

set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Avoid excessive thread overhead for these short sentences on CPU.
if DEVICE.type == "cpu":
    torch.set_num_threads(min(4, torch.get_num_threads()))
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)
print(f"Device: {DEVICE}; torch: {torch.__version__}; transformers: {transformers.__version__}")

Device: cpu; torch: 2.11.0+cpu; transformers: 4.57.6


## 1. The ten English inputs

In [3]:
english_texts = [
    "Hi, what's up?",
    "Can you give me a hand?",
    "No way!",
    "I gotta go now, see ya later.",
    "Long time no see!",
    "Hi, what's up?",
    "Can you give me a hand?",
    "No way!",
    "I gotta go now, see ya later.",
    "Long time no see!",  # Added to complete the original repeated set.
]
assert len(english_texts) == 10
inputs_df = pd.DataFrame({"input_id": range(1, 11), "english": english_texts})
display(inputs_df)

,input_id,english
0,1,"Hi, what's up?"
1,2,Can you give me a hand?
2,3,No way!
3,4,"I gotta go now, see ya later."
4,5,Long time no see!
5,6,"Hi, what's up?"
6,7,Can you give me a hand?
7,8,No way!
8,9,"I gotta go now, see ya later."
9,10,Long time no see!


## 2. Load the English-to-French model and define translation

The bilingual model accepts plain English. The multilingual ROMANCE model needs a target
prefix such as `>>fr<<`. Tokenization returns both input IDs and attention masks;
`generate(**inputs)` receives both, replacing the obsolete `prepare_translation_batch` call.

With sampling disabled, `num_beams=1` means greedy decoding. Larger values search several
candidate sequences. For beam search, `num_return_sequences` must not exceed `num_beams`.
Generation is limited to 80 new tokens, which is ample for these short inputs.

In [4]:
def load_marian(model_name):
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(DEVICE)
    model.eval()
    return tokenizer, model


def translate(texts, tokenizer, model, *, pair, num_beams=4, n_best=1, target_code=None):
    if not texts or any(not isinstance(text, str) or not text.strip() for text in texts):
        raise ValueError("Provide non-empty input sentences.")
    if num_beams < 1 or not 1 <= n_best <= num_beams:
        raise ValueError("Require 1 <= n_best <= num_beams.")
    if target_code is not None:
        prefix = f">>{target_code}<<"
        if prefix not in tokenizer.supported_language_codes:
            raise ValueError(f"Unsupported target language: {target_code}")
        source_texts = [f"{prefix} {text}" for text in texts]
    else:
        source_texts = texts
    inputs = tokenizer(source_texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    settings = dict(
        num_beams=num_beams,
        num_return_sequences=n_best,
        do_sample=False,
        max_new_tokens=80,
        length_penalty=1.0,
        return_dict_in_generate=True,
        output_scores=True,
    )
    if num_beams > 1:
        settings["early_stopping"] = True
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(**inputs, **settings)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    hypotheses = tokenizer.batch_decode(generated.sequences, skip_special_tokens=True)
    sequence_scores = getattr(generated, "sequences_scores", None)
    assert len(hypotheses) == len(texts) * n_best
    rows = []
    # Hugging Face groups each input's returned sequences together, best first.
    for input_index, english in enumerate(texts):
        for candidate_index in range(n_best):
            output_index = input_index * n_best + candidate_index
            rows.append({
                "input_id": input_index + 1,
                "english": english,
                "language_pair": pair,
                "model": model.config._name_or_path,
                "num_beams": num_beams,
                "rank": candidate_index + 1,
                "translation": hypotheses[output_index],
                "sequence_score": (
                    float(sequence_scores[output_index].item())
                    if sequence_scores is not None else float("nan")
                ),
            })
    return pd.DataFrame(rows), elapsed


fr_tokenizer, fr_model = load_marian("Helsinki-NLP/opus-mt-en-fr")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/301M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/301M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

## 3. English → French: compare beam widths

All ten sentences use the same model and decoding settings except `num_beams`.
The timing covers generation for the whole batch, excluding model loading and tokenization.
It is a single-run illustration, not a benchmark (the first run can include warm-up overhead).
Wider beams can produce the same translation and do not guarantee better quality.

In [5]:
BEAM_VALUES = [1, 2, 4, 8]
beam_frames, timing_rows = [], []
for beams in BEAM_VALUES:
    frame, seconds = translate(
        english_texts, fr_tokenizer, fr_model,
        pair="English → French", num_beams=beams,
    )
    beam_frames.append(frame)
    timing_rows.append({"num_beams": beams, "batch_seconds": seconds})
beam_results = pd.concat(beam_frames, ignore_index=True)
beam_comparison = beam_results.pivot(
    index=["input_id", "english"], columns="num_beams", values="translation"
).rename(columns={b: f"beam_{b}" for b in BEAM_VALUES}).reset_index()
display(beam_comparison)
display(pd.DataFrame(timing_rows).round(3))

num_beams,input_id,english,beam_1,beam_2,beam_4,beam_8
0,1,"Hi, what's up?","Salut, quoi de neuf ?","Salut, quoi de neuf ?","Salut, quoi de neuf ?","Salut, quoi de neuf ?"
1,2,Can you give me a hand?,Tu peux m'aider ?,Tu peux m'aider ?,Tu peux m'aider ?,Tu peux m'aider ?
2,3,No way!,Pas moyen !,C'est pas vrai !,Pas moyen !,Pas moyen !
3,4,"I gotta go now, see ya later.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure."
4,5,Long time no see!,Ça fait longtemps !,Ça fait longtemps !,Ça fait longtemps !,Ça fait longtemps !
5,6,"Hi, what's up?","Salut, quoi de neuf ?","Salut, quoi de neuf ?","Salut, quoi de neuf ?","Salut, quoi de neuf ?"
6,7,Can you give me a hand?,Tu peux m'aider ?,Tu peux m'aider ?,Tu peux m'aider ?,Tu peux m'aider ?
7,8,No way!,Pas moyen !,C'est pas vrai !,Pas moyen !,Pas moyen !
8,9,"I gotta go now, see ya later.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure.","Je dois y aller, à tout à l'heure."
9,10,Long time no see!,Ça fait longtemps !,Ça fait longtemps !,Ça fait longtemps !,Ça fait longtemps !


,num_beams,batch_seconds
0,1,1.786
1,2,3.840
2,4,3.200
3,8,4.143


## 4. N-best hypotheses for English → French

Return three candidates per input for beam widths 4 and 8: **60 rows** in total.
Rank 1 is the best candidate found by that search. The sequence score is a length-normalized
model log score; higher (usually less negative) is better **according to the model**.
It is not a probability or a translation-quality metric. Compare scores only for the same
input, model, and scoring settings. Greedy decoding has no beam sequence score and is
represented by a missing value in the earlier results. Distinct token sequences may decode
to identical visible text; this is not diverse beam search.

In [6]:
N_BEST = 3
nbest_frames = []
for beams in [4, 8]:
    frame, _ = translate(
        english_texts, fr_tokenizer, fr_model,
        pair="English → French", num_beams=beams, n_best=N_BEST,
    )
    nbest_frames.append(frame)
nbest_results = pd.concat(nbest_frames, ignore_index=True)
for beams in [4, 8]:
    print(f"Top {N_BEST} hypotheses with num_beams={beams}")
    display(nbest_results.loc[nbest_results.num_beams == beams, [
        "input_id", "english", "rank", "translation", "sequence_score"
    ]].reset_index(drop=True))

Top 3 hypotheses with num_beams=4


,input_id,english,rank,translation,sequence_score
0,1,"Hi, what's up?",1,"Salut, quoi de neuf ?",-0.337003
1,1,"Hi, what's up?",2,"Salut, quoi de neuf?",-0.523718
2,1,"Hi, what's up?",3,"Bonjour, quoi de neuf ?",-0.525243
3,2,Can you give me a hand?,1,Tu peux m'aider ?,-0.316903
4,2,Can you give me a hand?,2,Pouvez-vous m'aider ?,-0.401221
5,2,Can you give me a hand?,3,Vous pouvez m'aider ?,-0.448657
6,3,No way!,1,Pas moyen !,-0.636062
7,3,No way!,2,Pas moyen!,-0.807452
8,3,No way!,3,Pas du tout !,-0.818347
9,4,"I gotta go now, see ya later.",1,"Je dois y aller, à tout à l'heure.",-0.364349


Top 3 hypotheses with num_beams=8


,input_id,english,rank,translation,sequence_score
0,1,"Hi, what's up?",1,"Salut, quoi de neuf ?",-0.337003
1,1,"Hi, what's up?",2,"Salut, quoi de neuf?",-0.523718
2,1,"Hi, what's up?",3,"Bonjour, quoi de neuf ?",-0.525244
3,2,Can you give me a hand?,1,Tu peux m'aider ?,-0.316903
4,2,Can you give me a hand?,2,Pouvez-vous m'aider ?,-0.401222
5,2,Can you give me a hand?,3,Vous pouvez m'aider ?,-0.448657
6,3,No way!,1,Pas moyen !,-0.636061
7,3,No way!,2,Pas moyen!,-0.807452
8,3,No way!,3,Pas du tout !,-0.818348
9,4,"I gotta go now, see ya later.",1,"Je dois y aller, à tout à l'heure.",-0.364349


In [7]:
del fr_model, fr_tokenizer
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
romance_tokenizer, romance_model = load_marian("Helsinki-NLP/opus-mt-en-ROMANCE")
TARGET_LANGUAGES = {"fr": "French", "es": "Spanish", "it": "Italian", "pt": "Portuguese"}
language_frames = []
for code, language in TARGET_LANGUAGES.items():
    frame, _ = translate(
        english_texts, romance_tokenizer, romance_model,
        pair=f"English → {language}", num_beams=4, target_code=code,
    )
    language_frames.append(frame)
language_results = pd.concat(language_frames, ignore_index=True)
language_comparison = language_results.pivot(
    index=["input_id", "english"], columns="language_pair", values="translation"
).reset_index()
display(language_comparison)

tokenizer_config.json:   0%|          | 0.00/265 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/779k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/799k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

language_pair,input_id,english,English → French,English → Italian,English → Portuguese,English → Spanish
0,1,"Hi, what's up?","Salut, quoi de neuf ?","Ciao, che succede?","Olá, o que se passa?","Hola, ¿qué pasa?"
1,2,Can you give me a hand?,Pouvez-vous m'aider ?,Puoi darmi una mano?,Podes ajudar-me?,¿Puedes ayudarme?
2,3,No way!,Pas moyen !,Assolutamente no!,Nem pensar!,¡De ninguna manera!
3,4,"I gotta go now, see ya later.","Je dois y aller, à plus tard.","Devo andare, ci vediamo dopo.","Tenho de ir agora, até logo.","Tengo que irme ahora, hasta luego."
4,5,Long time no see!,Ça fait longtemps que je ne te vois pas !,Da quanto tempo non ci vediamo!,Há quanto tempo não nos vemos!,¡Hace mucho que no nos vemos!
5,6,"Hi, what's up?","Salut, quoi de neuf ?","Ciao, che succede?","Olá, o que se passa?","Hola, ¿qué pasa?"
6,7,Can you give me a hand?,Pouvez-vous m'aider ?,Puoi darmi una mano?,Podes ajudar-me?,¿Puedes ayudarme?
7,8,No way!,Pas moyen !,Assolutamente no!,Nem pensar!,¡De ninguna manera!
8,9,"I gotta go now, see ya later.","Je dois y aller, à plus tard.","Devo andare, ci vediamo dopo.","Tenho de ir agora, até logo.","Tengo que irme ahora, hasta luego."
9,10,Long time no see!,Ça fait longtemps que je ne te vois pas !,Da quanto tempo non ci vediamo!,Há quanto tempo não nos vemos!,¡Hace mucho que no nos vemos!


## 6. Check coverage and save the generated results

The assertions check experiment coverage and candidate ranks, not linguistic correctness.
CSV files are written to `translation_results/` relative to the notebook's working directory.

In [8]:
assert len(beam_results) == 10 * len(BEAM_VALUES)
assert beam_results.groupby("num_beams").input_id.nunique().eq(10).all()
assert len(nbest_results) == 10 * 2 * N_BEST
assert nbest_results.groupby(["num_beams", "input_id"])["rank"].apply(list).apply(
    lambda ranks: ranks == [1, 2, 3]
).all()
assert nbest_results.sequence_score.notna().all()
assert len(language_results) == 10 * len(TARGET_LANGUAGES)
assert language_results.groupby("language_pair").input_id.nunique().eq(10).all()
for frame in [beam_results, nbest_results, language_results]:
    assert frame.translation.str.strip().ne("").all()

output_dir = Path("translation_results")
output_dir.mkdir(exist_ok=True)
for filename, frame in {
    "english_french_beam_search.csv": beam_results,
    "english_french_nbest.csv": nbest_results,
    "different_language_pairs.csv": language_results,
    "beam_timings.csv": pd.DataFrame(timing_rows),
}.items():
    frame.to_csv(output_dir / filename, index=False)
print(f"All coverage checks passed. Results saved to {output_dir.resolve()}")

All coverage checks passed. Results saved to /content/translation_results


## Interpreting the output

- Compare “Can you give me a hand?” and “No way!” across hypotheses: does the translation
  preserve the conversational meaning rather than translating each word literally?
- Check whether the informal tone of “gotta” and “see ya” survives in French.
- Inspect where beam widths change the top translation and whether you prefer the change.
  Repeated inputs should provide a useful consistency check with sampling disabled.
- Automatic quality claims require reference translations or human evaluation; none are
  provided here, so no BLEU score or claim of accuracy is made.

Optional extension from the original assignment: fine-tune on
`conversational_en_fr_colloquial.csv` with LoRA and compare before/after outputs. Fine-tuning
is separate from the translation experiments implemented here.

References: [MarianMT usage](https://huggingface.co/docs/transformers/model_doc/marian),
[generation parameters](https://huggingface.co/docs/transformers/main_classes/text_generation),
[ROMANCE model and target prefixes](https://huggingface.co/Helsinki-NLP/opus-mt-en-ROMANCE).